In [2]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

In [3]:
df = pd.read_csv("data/spotify_2015_2025_85k.csv")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)


release_col = "release_date"
genre_col = "genre"
stream_col = "stream_count"


df["year"] = pd.to_datetime(df[release_col], errors="coerce").dt.year


df = df[["year", genre_col, stream_col]].copy()
df = df.rename(columns={
    genre_col: "genre",
    stream_col: "stream_count"
})


df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["stream_count"] = pd.to_numeric(df["stream_count"], errors="coerce")
df["genre"] = df["genre"].astype(str).str.strip()

df = df.dropna(subset=["year", "genre", "stream_count"])
df = df[df["genre"] != ""]
df = df[~df["genre"].str.lower().isin(["unknown", "none", "nan"])]
df = df[df["year"].between(2015, 2025)]

In [4]:
all_genres_df = (
    df.groupby(['year', 'genre'], as_index=False)['stream_count']
    .mean()
    .sort_values(['genre', 'year'])
)
all_genres = sorted(all_genres_df['genre'].unique())

base_color = "lightblue"
highlight_color = "blue"

fig = go.Figure()

annotation_texts = {genre: 'Punto máximo de reproducciones promedio del género.' for genre in all_genres}

for genre in all_genres:
    genre_data = all_genres_df[all_genres_df['genre'] == genre]
    max_row = genre_data.loc[genre_data['stream_count'].idxmax()]
    min_row = genre_data.loc[genre_data['stream_count'].idxmin()]
    max_year = max_row['year']
    min_year = min_row['year']

    fig.add_trace(go.Scatter(
        x=genre_data['year'],
        y=genre_data['stream_count'],
        mode='lines',
        name=genre,
        line=dict(shape='linear', width=3, color=base_color),
        hoverinfo='y',
        visible=True,
        legendgroup=genre,
        opacity=1
    ))

    fig.add_trace(go.Scatter(
        x=[max_row['year']],
        y=[max_row['stream_count']],
        mode='markers',
        name=genre,
        marker=dict(size=8, color=base_color),
        showlegend=False,
        hoverinfo='skip',
        visible=True,
        legendgroup=genre,
        opacity=1
    ))

    fig.add_trace(go.Scatter(
        x=[min_row['year']],
        y=[min_row['stream_count']],
        mode='markers',
        name=genre,
        marker=dict(size=8, color=base_color),
        showlegend=False,
        hoverinfo='skip',
        visible=True,
        legendgroup=genre,
        opacity=1
    ))

    for _, row in genre_data.iterrows():
        if row['year'] not in [2022, max_year, min_year]:
            fig.add_trace(go.Scatter(
                x=[row['year']],
                y=[row['stream_count']],
                mode='markers',
                name=genre,
                marker=dict(size=8, color=base_color),
                showlegend=False,
                hoverinfo='skip',
                visible=True,
                legendgroup=genre,
                opacity=1
            ))

    last_point = genre_data[genre_data['year'] == 2025]

    fig.add_trace(go.Scatter(
        x=[2025],
        y=[last_point['stream_count'].iloc[0]],
        mode='text',
        text=[f'    {genre}'],
        textposition='middle right',
        textfont=dict(size=13, color=base_color),
        showlegend=False,
        hoverinfo='skip',
        visible=True,
        legendgroup=genre,
        opacity=1
    ))

fig.update_layout(
    title="Evolución del Consumo Promedio de los Géneros Musicales (2015–2025)",
    xaxis=dict(
        title="Año",
        tickmode="linear",
        dtick=1,
        type="linear"
    ),
    yaxis=dict(
        title="Consumo promedio por género (stream count)",
        showgrid=True,
        gridcolor="rgba(0,0,0,0.12)",
        zeroline=True,
        tickformat="~s",
        range=[0, None]
    ),
    template="simple_white",
    width=1600,
    height=800,
    margin=dict(l=80, r=320, t=160, b=110),
    plot_bgcolor="white",
    paper_bgcolor="white",
    dragmode=False,
    showlegend=False,
    font=dict(family="Arial, sans-serif", size=13)
)

fig.add_annotation(
    text="Datos basados en el promedio anual de streams por género a nivel global.",
    xref="paper",
    yref="paper",
    x=0.5,
    y=1.10,
    showarrow=False,
    align="center",
    font=dict(size=16, color="gray")
)

fig.add_annotation(
    text=(
        "Cada línea representa el consumo promedio anual, no valores acumulados.<br>"
        "Fuente: Spotify Music Analytics Dataset (2015–2025) / Rohit kumar"
    ),
    xref="paper",
    yref="paper",
    x=0,
    y=-0.13,
    showarrow=False,
    align="left",
    font=dict(size=11, color="gray")
)

# Ejemplos de mensajes específicos para algunos géneros, falta llenar para todos los géneros presentes en el dataset
genre_messages_dict = {
    "Pop": "El <b>Pop</b> domina las listas de streaming globales con ritmos pegadizos.<br>",
    "Rock": "El <b>Rock</b> mantiene una base de oyentes leales con un consumo muy estable a lo largo de los años.",
    "Hip Hop": "El <b>Hip Hop</b> ha mostrado el mayor crecimiento porcentual desde 2018.",
}

for g in all_genres:
    if g not in genre_messages_dict:
        genre_messages_dict[g] = f"Información faltante para <b>{g}</b>."

html_content = f"""
<html>
<head>
  <meta charset='utf-8' />
  <script src='https://cdn.plot.ly/plotly-2.29.0.min.js'></script>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 0; padding: 20px; background: white; }}
    #plot {{ min-height: 640px; }}
    #info-box {{
      position: absolute;
      top: 20px;
      right: 20px;
      width: 300px;
      max-width: 320px;
      min-height: 160px;
      padding: 14px;
      border-radius: 12px;
      border: 1px solid rgba(0,0,0,0.12);
      background: rgba(255,255,255,0.96);
      box-shadow: 0 14px 28px rgba(0,0,0,0.14);
      display: none;
      font-family: Arial, sans-serif;
      color: #222;
      z-index: 10;
      box-sizing: border-box;
    }}
    #info-box h3 {{
      margin: 0 0 12px;
      font-size: 18px;
      font-weight: 600;
      color: #333;
    }}
    #genre-info {{
      width: 100%;
      min-height: 120px;
      padding: 10px;
      border: 1px solid rgba(0,0,0,0.16);
      border-radius: 8px;
      font-size: 14px;
      font-family: Arial, sans-serif;
      background-color: #f9f9f9;
      box-sizing: border-box;
      overflow-y: auto;
    }}
  </style>
</head>
<body>
  <div id='plot' style='position: relative;'></div>
  
  <div id='info-box'>
    <h3 id='genre-name'>Seleccione un género</h3>
    <div id='genre-info'>Aquí aparecerá la información...</div>
  </div>

  <script>
    const fig = {json.dumps(fig.to_plotly_json())};
    const baseColor = '{base_color}';
    const highlightColor = '{highlight_color}';
    
    // 3. INYECTAMOS EL DICCIONARIO COMO UN OBJETO JAVASCRIPT
    const genreMessages = {json.dumps(genre_messages_dict)};

    const plotDiv = document.getElementById('plot');
    const infoBox = document.getElementById('info-box');
    const genreName = document.getElementById('genre-name');
    const genreInfo = document.getElementById('genre-info');

    let currentSelectedGenre = null;

    Plotly.newPlot(plotDiv, fig.data, fig.layout, {{
      responsive: true,
      displayModeBar: true,
      modeBarButtonsToRemove: ['select2d', 'lasso2d', 'zoom2d', 'zoomIn2d', 'zoomOut2d', 'autoScale2d'],
      scrollZoom: false
    }});

    function showInfoBox(genre) {{
      genreName.textContent = genre;
      
      // 4. BUSCAMOS EL MENSAJE ESPECÍFICO DEL GÉNERO
      // (Si por alguna razón no existe en el diccionario, usamos un texto de respaldo)
      genreInfo.innerHTML = genreMessages[genre] || 'Información no disponible para este género.'; 
      
      infoBox.style.display = 'block';
    }}

    plotDiv.on('plotly_click', function(data) {{
      if (data.points.length > 0) {{
        const clickedTrace = fig.data[data.points[0].curveNumber];
        const clickedGenre = clickedTrace.legendgroup;

        if (currentSelectedGenre === clickedGenre) {{
          currentSelectedGenre = null; 
          infoBox.style.display = 'none'; 

          fig.data.forEach(trace => {{
            trace.opacity = 1;
            if (trace.mode === 'lines') {{
              trace.line.color = baseColor;
            }} else if (trace.mode === 'markers') {{
              if (trace.marker.line) {{
                trace.marker.line.color = baseColor;
              }} else {{
                trace.marker.color = baseColor;
              }}
            }} else if (trace.mode === 'text') {{
              trace.textfont.color = baseColor;
            }}
          }});
        }} 
        else {{
          currentSelectedGenre = clickedGenre; 
          showInfoBox(clickedGenre);

          fig.data.forEach(trace => {{
            if (trace.legendgroup === clickedGenre) {{
              trace.opacity = 1;
              if (trace.mode === 'lines') {{
                trace.line.color = highlightColor;
              }} else if (trace.mode === 'markers') {{
                if (trace.marker.line) {{
                  trace.marker.line.color = highlightColor;
                }} else {{
                  trace.marker.color = highlightColor;
                }}
              }} else if (trace.mode === 'text') {{
                trace.textfont.color = highlightColor;
              }}
            }} else {{
              trace.opacity = 0.18;
              if (trace.mode === 'lines') {{
                trace.line.color = baseColor;
              }} else if (trace.mode === 'markers') {{
                if (trace.marker.line) {{
                  trace.marker.line.color = baseColor;
                }} else {{
                  trace.marker.color = baseColor;
                }}
              }} else if (trace.mode === 'text') {{
                trace.textfont.color = baseColor;
              }}
            }}
          }});
        }}
        
        Plotly.react(plotDiv, fig.data, fig.layout, {{responsive: true}});
      }}
    }});
  </script>
</body>
</html>
"""

output_path = os.path.join("..", "docs", "index.html")

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)